# 🔍 05 · 跨模态检索与评估

> 多模态模型上线怎么评估？**图文检索（text-to-image / image-to-text）** 是最直接的任务。
> 本节手写跨模态检索 top-k、Recall@K、zero-shot 分类准确率，并演示 CLIP score 思想。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 图文检索怎么评估？ | Recall@K |
| 2 | 检索怎么实现？ | embedding 相似度 top-k |
| 3 | zero-shot 迁移怎么测？ | 模板+分类准确率 |
| 4 | 生成质量怎么评？ | CLIP score / 人工偏好 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 构造图文 embedding 测试集

In [2]:
n = 500
# 确定性共享向量 + 可调噪声：控制检索难度
shared = rng.normal(0, 1, (n, 64))
img_emb = shared + rng.normal(0, 1.0, (n, 64))
txt_emb = shared + rng.normal(0, 1.0, (n, 64))
img_emb /= np.linalg.norm(img_emb, axis=1, keepdims=True)
txt_emb /= np.linalg.norm(txt_emb, axis=1, keepdims=True)
print(f'{n} 个图文对，embedding 维度 64')
print('➡️ 检索目标：给定一个文本，找出最相似的图像（反之亦然）')

500 个图文对，embedding 维度 64
➡️ 检索目标：给定一个文本，找出最相似的图像（反之亦然）


## 2️⃣ 跨模态检索 top-k 与 Recall@K

In [3]:
sim = img_emb @ txt_emb.T   # (n, n) 图像-文本相似度

def recall_at_k(sim_matrix, k):
    # 对每个查询，看正确配对是否在 top-k
    n = sim_matrix.shape[0]
    hits = sum(1 for i in range(n) if i in np.argsort(-sim_matrix[i])[:k])
    return hits / n

for k in [1, 5, 10]:
    print(f'text->image Recall@{k:>2} = {recall_at_k(sim, k):.3f}')
r10 = recall_at_k(sim, 10)
assert r10 > 0.5
print('\n✅ 检索质量随 k 增大而提升（与直觉一致）')

text->image Recall@ 1 = 0.918
text->image Recall@ 5 = 0.986
text->image Recall@10 = 0.992

✅ 检索质量随 k 增大而提升（与直觉一致）


## 3️⃣ 加噪后的检索退化（评估敏感性）

In [4]:
# 加大扰动看检索退化（在共享向量基础上加不同强度噪声）
for noise_scale in [0.5, 1.0, 2.0]:
    img2 = shared + rng.normal(0, noise_scale, (n, 64))
    img2 /= np.linalg.norm(img2, axis=1, keepdims=True)
    sim2 = img2 @ txt_emb.T
    r = recall_at_k(sim2, 5)
    print(f'噪声尺度 {noise_scale}: Recall@5 = {r:.3f}')
print('\n➡️ 评估要覆盖不同难度：模型在"干净/带噪/难负样本"下都要测')

噪声尺度 0.5: Recall@5 = 1.000
噪声尺度 1.0: Recall@5 = 0.970
噪声尺度 2.0: Recall@5 = 0.604

➡️ 评估要覆盖不同难度：模型在"干净/带噪/难负样本"下都要测


## 4️⃣ Zero-shot 分类准确率（CLIP 评估）

In [5]:
# 模拟 10 类分类：每类 50 张图，类名文本模板与图相似度（true 类最高）
n_cls, per = 10, 50
label = np.repeat(np.arange(n_cls), per)
# 每类一个"类级别"共享向量：类中心图像与类名模板共享它 -> 同类相似度高
cls_vec = rng.normal(0, 1, (n_cls, 64))
img_cls = cls_vec + rng.normal(0, 0.5, (n_cls, 64))     # 类中心图像
txt_template = cls_vec + rng.normal(0, 0.5, (n_cls, 64))  # 类名文本模板
img_cls /= np.linalg.norm(img_cls, axis=1, keepdims=True)
txt_template /= np.linalg.norm(txt_template, axis=1, keepdims=True)
sim_cls = img_cls @ txt_template.T

# 每张图与自己类的模板相似度最高
sim_cls = img_cls @ txt_template.T
# 给每个样本分配相似度（同类模板分高）
scores = np.zeros((n_cls * per, n_cls))
for i in range(n_cls):
    for j in range(per):
        scores[i*per + j] = sim_cls[i] + rng.normal(0, 0.2, n_cls)
pred = scores.argmax(axis=1)
acc = (pred == label).mean()
print(f'Zero-shot 分类准确率: {acc:.1%}')
assert acc > 0.9
print('✅ zero-shot 分类评估完成（无任何训练）')

Zero-shot 分类准确率: 95.8%
✅ zero-shot 分类评估完成（无任何训练）


## 5️⃣ 生成质量评估：CLIP score 思想

In [6]:
print('''CLIP score：生成图像与文本描述的余弦相似度，衡量"图文相关性"。
  优点：免人工、快；缺点：不等于人类偏好（可能风格差但相关）。
其他评估：
  图像质量：FID/IS（分布距离/多样性）；
  图文相关性：CLIP score / BLIP-VQA；
  人类偏好：A/B 对比 / Reward model；
  多模态基准：MMBench、MMMU、SEED-Bench、MM-Vet。
''')

CLIP score：生成图像与文本描述的余弦相似度，衡量"图文相关性"。
  优点：免人工、快；缺点：不等于人类偏好（可能风格差但相关）。
其他评估：
  图像质量：FID/IS（分布距离/多样性）；
  图文相关性：CLIP score / BLIP-VQA；
  人类偏好：A/B 对比 / Reward model；
  多模态基准：MMBench、MMMU、SEED-Bench、MM-Vet。



## 6️⃣ 多模态评测陷阱（面试加分点）

In [7]:
print('''1) 数据泄漏：预训练数据与评测集重叠 → 分数虚高；
2) 提示差异：prompt 模板不同结果差很大（敏感度检验）；
3) 语言偏置：英文基准强不代表中文强；
4) 任务偏置：只测 OCR/计数等单一能力会误导；
5) 人工评测偏差：评判标准不一致、锚定效应。
''')

1) 数据泄漏：预训练数据与评测集重叠 → 分数虚高；
2) 提示差异：prompt 模板不同结果差很大（敏感度检验）；
3) 语言偏置：英文基准强不代表中文强；
4) 任务偏置：只测 OCR/计数等单一能力会误导；
5) 人工评测偏差：评判标准不一致、锚定效应。



## 📝 自测清单

- [ ] 能手写 Recall@K 并解释含义
- [ ] 能演示噪声对检索质量的影响
- [ ] 能做 zero-shot 分类准确率评估
- [ ] 能说出 CLIP score、FID、人类偏好等评估手段
- [ ] 能列举多模态评测五大陷阱